In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    # base_url=os.getenv("OLLAMA_BASE_URL"),
    # model="gemma4:31b-cloud",
    model="gemma4:e2b",
    temperature=0.7,
    # client_kwargs={
    #     "headers": {
    #         "Authorization": f"Bearer {os.getenv('OLLAMA_API_KEY')}"
    #     }
    # }
)

model.invoke("Hi How are you?").content

In [ ]:
from langchain.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


tools = [add, multiply, divide]

tools_by_name = {tool.name: tool for tool in tools}

In [ ]:
model = model.bind_tools(tools)

In [ ]:
from typing_extensions import TypedDict, Annotated
import operator

from langchain_core.messages import AnyMessage


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [ ]:
from langchain_core.messages import ToolMessage


def model_node(state: MessagesState) -> MessagesState:
    """
        Model decides whether to call a tool or not
    """

    query = state["messages"][-1]
    print(f"user query: {query.content}")

    response = model.invoke(state["messages"])

    return {
        "messages": [response],
        "llm_calls": state["llm_calls"] + 1
    }


def tool_node(state: MessagesState) -> MessagesState:
    """
        Performs a tool call
    """

    print(state["messages"][-1].tool_calls)

    result: list[ToolMessage] = []

    tool_calls = state["messages"][-1].tool_calls

    for tool_call in tool_calls:
        tool_name = tool_call["name"]

        tool = tools_by_name[tool_name]

        tool_result = tool.invoke(tool_call["args"])

        result.append(ToolMessage(
            content=tool_result, tool_call_id=tool_call["id"]))

    return {
        "messages": result
    }

In [ ]:
from typing import Literal
from langgraph.graph import END


def should_continue(state: MessagesState) -> Literal["tool_node", "__end__"]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tool_node"

    return END

In [ ]:
from langgraph.graph import StateGraph, START

graph = StateGraph(MessagesState)

graph.add_node("llm_call", model_node)
graph.add_node("tool_node", tool_node)


graph.add_edge(START, "llm_call")
graph.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)

graph.add_edge("tool_node", "llm_call")

app = graph.compile()
app

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AnyMessage

system_instruction = SystemMessage(
    content="""
        You are a helpful assistant.
    """
)

messages: list[AnyMessage] = [system_instruction]

while True:
    user_query = input("Enter you query !")

    messages.append(HumanMessage(content=user_query))

    response = app.invoke({"messages": messages, "llm_calls": 0})

    messages = response["messages"]

    messages[-1].pretty_print()